In [ ]:
import sys
sys.path.insert(1, '../')
import src.data.utils as ut
from src.model.inference import PanInference
from src.model.train import get_mask_trans
import os
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
from io import BytesIO
import pandas as pd
from tqdm import tqdm
from torchmetrics.segmentation import MeanIoU
import torch
from src.data.utils import IDX2CAT

        
def save_pil_image(img, folder_path, filename,verbose=False):
    """
    Saves a single PIL image to the specified folder.
    Converts RGBA to RGB if necessary for saving as JPEG.

    :param img: PIL Image to save
    :param folder_path: Folder where the image should be saved
    :param filename: Name of the image file (e.g., 'image.png')
    """
    # Create the folder if it doesn't exist
    os.makedirs(folder_path, exist_ok=True)

    # Convert RGBA to RGB if necessary (JPEG does not support RGBA)
    if img.mode == 'RGBA':
        img = img.convert('RGB')

    # Create the full path
    img_path = os.path.join(folder_path, filename)

    # Save the image (you can specify the format if needed, e.g., 'JPEG' or 'PNG')
    img.save(img_path, format='JPEG')  # Save as JPEG if desired, or use PNG if transparency is needed
    if verbose:
        print(f"Saved image to: {img_path}")

def plot_pred_true_image(pred_img, true_img, fig_size=(10, 5), return_pil=False):
    """
    Plots two PIL images side by side for comparison.
    
    :param pred_img: PIL Image (Predicted image)
    :param true_img: PIL Image (Ground truth image)
    :param fig_size: Tuple (Width, Height) for figure size
    :param return_pil: If True, returns the plot as a PIL image instead of displaying it
    :return: PIL Image if return_pil=True, else None
    """
    fig, axes = plt.subplots(1, 2, figsize=fig_size)

    # Plot images
    axes[0].imshow(pred_img)
    axes[0].set_title("Predicted Image")
    axes[0].axis("off")

    axes[1].imshow(true_img)
    axes[1].set_title("Ground Truth Image")
    axes[1].axis("off")

    plt.tight_layout()

    if return_pil:
        buf = BytesIO()
        plt.savefig(buf, format="png", bbox_inches="tight", pad_inches=0)
        plt.close(fig)  # Close the figure to free memory
        buf.seek(0)
        return Image.open(buf)  # Return as a PIL image
    else:
        plt.show()

# Downloading and Preprocessing data

In [ ]:
json_file_name = "impacted_canines.json"
img_folder_name = os.path.splitext(json_file_name)[0]
remote_folder = os.path.join("/home/anno/coco-annotator/datasets/",img_folder_name)
local_folder = os.path.join("../data/test_set/raw_imgs",img_folder_name)
remote_host = "<ANNOTATION_SERVER_HOST>"
remote_user="<USER>"



In [ ]:
ut.download_folder(
    remote_folder, local_folder, remote_host, remote_user
)

In [ ]:
manual_download_dir="data/test_set/downloaded_json/"
mask_output_dir="data/test_set/mask/"
raw_img_dir ="data/test_set/raw_imgs/"


In [ ]:
metadata_df = ut.get_processed_df(json_file_name,manual_download_dir,mask_output_dir,raw_img_dir)

In [ ]:
metadata_path = "../data/test_set/metadata/custom_test_set.csv"
metadata_df.to_csv(metadata_path)

# Calculate the human baseline error

In [ ]:
impacted_canines_dir_path = "../data/test_set/mask/impacted_canines"
file_names =os.listdir(impacted_canines_dir_path)
iou = MeanIoU(num_classes=33,per_class=True)

for fname in file_names:
    if fname.endswith(".png"):
        ic_mask = Image.open(os.path.join(impacted_canines_dir_path,fname))
        ic_mask = get_mask_trans(512,512)(ic_mask).long()
        ic1_mask = Image.open(os.path.join(impacted_canines_dir_path+"_1",fname))
        ic1_mask = get_mask_trans(512,512)(ic1_mask).long()
        iou(ic1_mask,ic_mask)
    else:
        print(f"{fname} is not a .png file.")
    


In [ ]:
iou_score = iou.compute()
iou_score[1:].mean()

In [ ]:

metrics_df = pd.DataFrame({
            "iou":iou_score
        },index=[IDX2CAT[i] for i in range(33)])
metrics_df.drop("BACKGROUND")

In [ ]:
df = pd.read_csv("../data/test_set/metadata/custom_test_set.csv")
df["set_name"]="impacted_canines_1"
df["mask_rel_path"] = df["mask_rel_path"].apply(lambda x:x.replace("impacted_canines","impacted_canines_1"))
df.iloc[0]["mask_rel_path"]

In [ ]:
df.to_csv("../data/test_set/metadata/custom_test_set_1.csv",index=False)


# Infer 

In [ ]:
state_dict_path ="s3://<YOUR_BUCKET>/artifacts/148418839595485829/69ea416e875d433ea5326bf38832cf6b/artifacts/models/model_epoch_74"
root_dir = ".."
model_name = "deeplabv3_resnet50"
num_classes=33
img_size = (512,512)

pi = PanInference(model_name,state_dict_path,root_dir,num_classes,img_size=img_size)

In [ ]:
csv_path = "../data/test_set/metadata/custom_test_set.csv"
# csv_path ="../data/metadata/test_data.csv"
# test_sets =["pan_set_1","pan_set_2","pan_set_3","pan_set_4","pan_set_5","pan_set_6","pan_set_7"]
batch_size = 16


In [ ]:
pi.calc_predictions(csv_path,batch_size)

In [ ]:
pi.metrics_df.drop("BACKGROUND").to_excel("../data/test_set/temp_pred_imgs/impacted_canines/impacted_canines_metrics.xlsx")

In [ ]:
pi.metrics

In [ ]:
img_path = "data/test_set/raw_imgs/impacted_canines_1/Unilateral impaction 7.png"
pred_img,true_img = pi.get_pred_true_img_mask(img_path)


In [ ]:
plot_pred_true_image(pred_img,true_img,(15,12),True)

In [ ]:
data_df = pd.read_csv(csv_path)

In [ ]:
out_folder_path = f"../data/test_set/temp_pred_imgs/{img_folder_name}"
for _,row in tqdm(data_df.iterrows(),total=len(data_df)):
    img_path = row["img_rel_path"]
    pred_img,true_img = pi.get_pred_true_img_mask(img_path)
    comb_img = plot_pred_true_image(pred_img,true_img,(15,12),True)
    file_name = os.path.basename(img_path)
    save_pil_image(pred_img,os.path.join(out_folder_path,"pred_imgs"),file_name)
    save_pil_image(true_img,os.path.join(out_folder_path,"true_imgs"),file_name)
    save_pil_image(comb_img,os.path.join(out_folder_path,"pred_true_comb_imgs"),file_name)
    
    
    